## PROTECTION USING LLM-GUARD

LLM-Guard is a scanner which we can use to check input and/or output from an AI model for sensitive data leakage.  It doesn't run in CoLab, so we'll run it directly using Jupyter Notebook.
  
  We'll start by  installing the llm-guard library.

In [ ]:
%pip install llm-guard
%pip install openai

We'll now look at an example of how we use llm-guard to scan our input.  Note we have our OpenAPI key set as an environment variable.  Let's load the libraries we need and set up our AI client handle using our key.

In [ ]:
import os
from openai import OpenAI
from llm_guard import scan_output, scan_prompt
from llm_guard.input_scanners import Anonymize, PromptInjection, TokenLimit, Toxicity
from llm_guard.output_scanners import Deanonymize, NoRefusal, Relevance, Sensitive
from llm_guard.vault import Vault

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

We can ignore the numpy version issue for our run. We'll also set up a vault to use for managing the data we need to anonymize.  We'll load four input scanners and four output scanners.  This takes some time to run first time as it has to download scanning models.

In [ ]:
vault = Vault()
input_scanners = [Anonymize(vault), Toxicity(), TokenLimit(), PromptInjection()]
output_scanners = [Deanonymize(vault), NoRefusal(), Relevance(), Sensitive()]

We'll now hard code a sensitive prompt and use the llm-guard scanner to check it.  We'll check whether we can run the query, and we'll display what the sanitized query looks like.

In [ ]:
prompt = f"""
Make an SQL insert statement to add a new user to our database. His name is John Doe and his email is test@test.com. His phone number is 555-123-4567.
His credit card number is 4567-8901-2345-6789 and he works in Test LLC.
"""

sanitized_prompt, results_valid, results_score = scan_prompt(input_scanners, prompt)
if any(results_valid.values()) is False:
    print(f"Prompt {prompt} is not valid, scores: {results_score}")
    exit(1)

print(f"Prompt: {sanitized_prompt}")

OK, we found an email address and redacted it. We also found a person element which is flagged as sensitive and redacted that.  A toxicity scann was run but did not find any toxic content. Note the phone numbers don't appear in the redacted prompt.

We can now send the redacted prompt to OpenAI.

In [ ]:
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": sanitized_prompt},
    ],
    temperature=0,
    max_tokens=512,
)
response_text = response.choices[0].message.content
print(response_text)

We can now run the output scanner to recover the redacted text and check the output. 

In [ ]:
sanitized_response_text, results_valid, results_score = scan_output(
    output_scanners, sanitized_prompt, response_text
)
if any(results_valid.values()) is False:
    print(f"Output {response_text} is not valid, scores: {results_score}")
    exit(1)

print(f"Output: {sanitized_response_text}\n")

OK, we've got our result and the originally redacted text has been added back in place of the redaction placeholders. We get a warning that sensitive data is in the reconstituted output results.
